# Reflect and Refine Pattern
### When quality review is a process requirement

Signals your process needs reflection loops:
- **No embedded human review** — the loop is the quality gate
- **Output quality is measurable** — clarity, completeness, accuracy
- **Cost of iteration < cost of a poor output**

### Architecture

```
Change Request → Drafter Agent ──────────────────────────┐
                      ▲                                  ▼
                      │                          QA Reviewer Agent
                      │                          (checks against standards)
                      │                                  │
                      │                          PASS? ──┬── Yes → Approved
                      │                                  │
                      └───── feedback + gaps ─────────── No
                         (max 3 iterations)
```

### Business Use Case: AnyComp Telecom Network Change Request Review

Every network change at AnyComp Telecom (firmware upgrades, config updates, hardware swaps) requires a formal change plan that passes 8 QA standards before the Change Advisory Board will approve execution. Engineers routinely fail the first review because they miss rollback timelines or skip the communication plan, and each human round-trip takes hours. Here, a Drafter Agent produces the plan and a QA Reviewer Agent checks it against all 8 standards, returning specific feedback on any failures. The drafter revises and resubmits automatically, up to 3 iterations. Most plans pass within 1–2 loops, producing a CAB-ready plan in seconds instead of hours.

## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip install -q strands-agents>=1.34.0 strands-agents-tools boto3 "typing_extensions>=4.12.0"

## Install Dependencies

**Restart the kernel** after installing, then continue.

## Setup: Change Request and QA Standards

In [ ]:
import json

REGION = 'us-east-1'
MAX_ITERATIONS = 3

CHANGE_REQUEST = {
    "ncr_id": "NCR-2026-0412",
    "title": "Firmware upgrade on East region cell towers",
    "submitted_by": "James Wilson, Network Engineer",
    "description": (
        "Upgrade firmware from v4.2.1 to v5.0.0 on 50 cell towers in the East region. "
        "The upgrade addresses critical security vulnerabilities and improves 5G handoff performance. "
        "Proposed maintenance window: Saturday 2026-04-20, 02:00-06:00 UTC."
    ),
    "affected_devices": 50,
    "region": "East",
    "priority": "high",
}

QA_STANDARDS = [
    "1. SCOPE: Must list all affected device IDs or a clear selection criteria",
    "2. RISK ASSESSMENT: Must rate risk (low/medium/high) with justification",
    "3. ROLLBACK PLAN: Must include specific rollback steps and estimated rollback time",
    "4. IMPACT ASSESSMENT: Must describe customer impact during and after the change",
    "5. COMMUNICATION PLAN: Must list stakeholders to notify and notification timeline",
    "6. MAINTENANCE WINDOW: Must confirm the window is outside peak hours and has CAB approval",
    "7. TESTING: Must describe pre-change validation and post-change verification steps",
    "8. DEPENDENCIES: Must list any dependencies on other teams or systems",
]

print(f'Change request: {CHANGE_REQUEST["ncr_id"]}')
print(f'QA standards: {len(QA_STANDARDS)} criteria')

## Implementation 1: Strands Agents

Two Strands agents in a while-loop. The drafter produces a change plan, the reviewer checks it against standards. If gaps are found, feedback goes back to the drafter. Max 3 iterations.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=REGION,
    max_tokens=4096,
)

drafter = Agent(
    model=model,
    system_prompt=(
        "You are a network change planning specialist at AnyComp Telecom. "
        "Given a change request, produce a detailed change plan covering: "
        "scope, risk assessment, rollback plan, impact assessment, communication plan, "
        "maintenance window confirmation, testing steps, and dependencies. "
        "If you receive reviewer feedback, address every gap specifically."
    ),
)

standards_text = '\n'.join(QA_STANDARDS)

reviewer = Agent(
    model=model,
    system_prompt=(
        "You are a QA reviewer for network change requests at AnyComp Telecom. "
        "Review the change plan against these standards:\n"
        f"{standards_text}\n\n"
        "For each standard, mark it PASS or FAIL with a brief explanation. "
        "At the end, give an overall verdict: APPROVED or NEEDS_REVISION. "
        "If NEEDS_REVISION, list the specific gaps that must be addressed."
    ),
)

# ── Reflect-and-refine loop ──────────────────────────────────────────────────
print("=" * 60)
print("STRANDS — Reflect and Refine")
print("=" * 60)

change_plan = None
feedback = None

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f"\n--- Iteration {iteration}/{MAX_ITERATIONS} ---")

    # Draft (or revise based on feedback)
    if feedback:
        drafter_prompt = f"Revise the change plan based on this reviewer feedback:\n\n{feedback}\n\nOriginal plan:\n{change_plan}"
    else:
        drafter_prompt = f"Create a change plan for:\n{json.dumps(CHANGE_REQUEST, indent=2)}"

    print("  Drafter working...")
    change_plan = str(drafter(drafter_prompt))

    # Review
    print("  Reviewer checking...")
    review = str(reviewer(f"Review this change plan:\n\n{change_plan}"))

    if "APPROVED" in review.upper() and "NEEDS_REVISION" not in review.upper():
        print(f"  ✅ APPROVED on iteration {iteration}")
        break
    else:
        print(f"  ⚠️  NEEDS_REVISION — feeding back to drafter")
        feedback = review
else:
    print(f"\n⏳ Max iterations ({MAX_ITERATIONS}) reached. Submitting for human review.")

print("\n" + "=" * 60)
print("FINAL CHANGE PLAN:")
print("=" * 60)
print(change_plan[:3000])
print("\nREVIEWER VERDICT:")
print(review[:1000])

## Key Takeaway

The reflect-and-refine loop is the quality gate the process would otherwise require a human to perform. The reviewer agent applies consistent standards every time — no fatigue, no shortcuts, no missed criteria. The max iteration cap (`MAX_ITERATIONS = 3`) prevents infinite loops and ensures the process either converges on quality or escalates to a human reviewer.

In production, track:
- **Average iterations to approval** — if it’s consistently hitting max, the drafter prompt needs work
- **Which standards fail most often** — tells you where the drafter is weakest
- **Time per iteration** — the cost of quality vs the cost of a bad output reaching production